# 构建一个实用的天气预报代理
1. 定义系统提示

系统提示定义了你的代理的角色和行为。保持它具体且可执行

In [1]:

SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

print(SYSTEM_PROMPT)
print(SYSTEM_PROMPT)

You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location.
You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location.


2. 创建工具

工具允许模型通过调用你定义的函数与外部系统交互。工具可以依赖于运行时上下文，并且还可以与代理内存交互。

In [2]:
from dataclasses import dataclass

from langchain.tools import tool, ToolRuntime


@tool
def get_weather_for_location(city):
    """Get weather for given location."""
    return f"It is always sunny in {city}!"


@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]):
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

3. 配置模型

为使用场景设置正确的参数来配置语言模型

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "deepseek-chat",
    temperature="0.5",
    timeout=10,
    max_tokens=1000
)

4. 定义响应格式

如果需要指定代理响应特定模式，可以定义一个结构化的响应格式


In [6]:
@dataclass
class ResponseFormat:
    """Response schema for agent."""
    punny_response: str
    weather_conditions: str | None = None


5. 添加记忆

为代理添加记忆功能，在交互中保持状态。这允许代理记住之前的对话和上下文

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()


6. 创建并运行代理

用所有组件组装代理并运行

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer
)

config = {"configurable":{"thread_id":1}}

response = agent.invoke(
    {"message":[{"role":"user","content":"what is the weather outside?"}]},
    config,
    context=Context(user_id="1")
)

print(response['structured_response'])

ResponseFormat(punny_response="Well, well, well, looks like Florida is living up to its sunny reputation! I'd say the forecast is looking quite bright - it's always sunny in Florida! I guess you could say the weather there is... wait for it... absolutely Florida-ting! No need to worry about rain putting a damper on your plans - the only thing falling from the sky there is sunshine and good vibes. It's the perfect weather for making some rays of sunshine in your day!", weather_conditions='It is always sunny in Florida!')
